In [1]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No API key was found ")
else:
    print("API key found and looks good so far!")

client = Groq(api_key= api_key)


API key found and looks good so far!


In [2]:
import base64
import re
import pymupdf as fitz
 
def extract_content(file_path):
    """
    Given a file path, extract text and images.
    - PDF  → extract text from each page + render each page as a PNG image
    - Image → encode directly as base64, return empty string for text
 
    Returns:
        text (str), images (list of base64 data URL strings)
    """
    if file_path is None:
        return "", []
 
    ext = file_path.split(".")[-1].lower()
 
    if ext == "pdf":
        return _from_pdf(file_path)
    else:
        return _from_image(file_path)
 
 
def _from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    images = []
 
    for i, page in enumerate(doc):
        text += f"\n--- Page {i + 1} ---\n"
        text += page.get_text()
 
        # Render page as 150 DPI PNG — readable for handwriting
        pix = page.get_pixmap(matrix=fitz.Matrix(150 / 72, 150 / 72))
        b64 = base64.standard_b64encode(pix.tobytes("png")).decode("utf-8")
        images.append(f"data:image/png;base64,{b64}")
 
    doc.close()
    return text, images
 
 
def _from_image(file_path):
    ext = file_path.split(".")[-1].lower()
    media_types = {
        "jpg": "image/jpeg", "jpeg": "image/jpeg",
        "png": "image/png",  "gif": "image/gif", "webp": "image/webp"
    }
    media_type = media_types.get(ext, "image/jpeg")
 
    with open(file_path, "rb") as f:
        b64 = base64.standard_b64encode(f.read()).decode("utf-8")
 
    return "", [f"data:{media_type};base64,{b64}"]


def _parse_scores(text):
    matches   = re.findall(r'(\d+)\s*/\s*(\d+)', text)
    awarded   = sum(int(m[0]) for m in matches)
    available = sum(int(m[1]) for m in matches)
    return awarded, available

In [3]:

def run_text_agent(guide_file, script_file):
    if guide_file is None or script_file is None:
        return 'Please upload both files before clicking Grade.', 0, 0

    guide_text, guide_images = extract_content(guide_file)
    script_text, script_images = extract_content(script_file)

    prompt = '''You are a strict but fair AI examiner grading written exam answers.
    You will receive a MARKING GUIDE and a STUDENT SCRIPT.

    For each question, think step by step:
    1. What does the marking guide require?
    2. What did the student write?
    3. Which key points did the student cover?
    4. What is missing?
    5. Award marks proportionally based on key points covered.

    IMPORTANT RULES:
    - Do NOT require word-for-word matching. Grade on meaning and understanding.
    - Award partial marks if the student covered some but not all key points.
    - Ignore spelling errors unless the question specifically tests spelling.

    Format your response exactly like this:
    Q[N]: [marks awarded]/[marks available]
    Reasoning: [one sentence showing your reasoning]
    Feedback: [one sentence of constructive feedback for the student]
    '''

    content = []

    if guide_text.strip():
        content.append({'type': 'text', 'text': f'=== MARKING GUIDE ===\n{guide_text}'})
    for img in guide_images:
        content.append({'type': 'image_url', 'image_url': {'url': img}})

    if script_text.strip():
        content.append({'type': 'text', 'text': f'=== STUDENT SCRIPT ===\n{script_text}'})
    for img in script_images:
        content.append({'type': 'image_url', 'image_url': {'url': img}})

    content.append({'type': 'text', 'text': prompt})

    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{'role': 'user', 'content': content}]
    )

    result_text = completion.choices[0].message.content
    awarded, available = _parse_scores(result_text)

    return result_text, awarded, available


In [4]:
def run_image_agent(guide_file, drawing_file):
    if guide_file is None or drawing_file is None:
        return 'Please upload both files before clicking Grade.', 0, 0, False

    _, guide_images   = extract_content(guide_file)
    _, drawing_images = extract_content(drawing_file)

    if not guide_images:
        return 'No reference diagram found in the marking guide.', 0, 0, False
    if not drawing_images:
        return 'No drawing found in the student file.', 0, 0, False

    reference_diagram = guide_images[-1]
    student_drawing   = drawing_images[-1]

    prompt = '''You are an expert diagram examiner grading student drawn diagrams.
    The first image is the REFERENCE DIAGRAM from the marking guide.
    The second image is the STUDENT-DRAWN DIAGRAM.

    Compare the two images carefully and provide:
    1. A DIAGRAM SCORE based on how many elements were correctly reproduced
    2. ELEMENTS IN REFERENCE: every key element visible in the reference diagram
    3. CORRECTLY REPRODUCED: elements the student got right
    4. MISSING OR INCORRECT: elements that are wrong or absent
    5. If the drawing is too unclear to grade, write CLARITY: UNCLEAR

    Format your response exactly like this:
    DIAGRAM SCORE: [marks awarded]/[marks available]

    ELEMENTS IN REFERENCE:
    - [element 1]
    - [element 2]

    CORRECTLY REPRODUCED:
    - [element 1]
    - [element 2]

    MISSING OR INCORRECT:
    - [element 1]
    - [element 2]

    CLARITY: CLEAR / UNCLEAR
    Feedback: [one sentence of constructive feedback for the student]
    '''

    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            'role': 'user',
            'content': [
                {
                    'type': 'image_url',
                    'image_url': {'url': reference_diagram}
                },
                {
                    'type': 'image_url',
                    'image_url': {'url': student_drawing}
                },
                {
                    'type': 'text',
                    'text': prompt
                }
            ]
        }]
    )

    result_text   = completion.choices[0].message.content
    awarded, available = _parse_scores(result_text)
    human_review  = 'UNCLEAR' in result_text.upper()

    return result_text, awarded, available, human_review

In [5]:
def run_final_boss_agent(text_guide_file,
                         text_result, text_awarded, text_available,
                         image_result, image_awarded, image_available,
                         human_review_needed):

    if text_result is None:
        return 'Please run Text Grading first before generating the final report.'

    total_awarded   = text_awarded + image_awarded
    total_available = text_available + image_available
    percentage      = round((total_awarded / total_available) * 100) if total_available > 0 else 0

    guide_text, guide_images = extract_content(text_guide_file) if text_guide_file else ('', [])

    prompt = f'''You are the final examiner producing an official student grading report.
    You have received results from two specialist agents:
    1. A Text Grading Agent — graded the written and theory answers
    2. An Image Grading Agent — graded the drawn diagram

    Your job:
    - Combine both results into one clean, professional final report
    - Include per-question scores, per-question feedback, total score, and overall feedback
    - If the image section was not submitted, note that only text was graded
    - Be professional, fair, and encouraging in tone

    The total score has already been calculated for you:
    TOTAL SCORE: {total_awarded}/{total_available}  ({percentage}%)

    Format your response exactly like this:
    ═══════════════════════════════════════
             STUDENT GRADING REPORT
    ═══════════════════════════════════════

    WRITTEN / THEORY SECTION
    ───────────────────────────────────────
    [Reformat the text agent results cleanly here]

    DIAGRAM SECTION
    ───────────────────────────────────────
    [Reformat the image agent results here, or write Not Submitted if skipped]

    ═══════════════════════════════════════
    TOTAL SCORE : {total_awarded}/{total_available}  ({percentage}%)
    GRADE       : [assign a grade based on the percentage]
    ═══════════════════════════════════════

    OVERALL FEEDBACK
    [Two to three sentences of general constructive feedback for the student]
    '''

    content = []

    if guide_text.strip():
        content.append({'type': 'text', 'text': f'=== MARKING GUIDE (for context) ===\n{guide_text}'})
    for img in guide_images:
        content.append({'type': 'image_url', 'image_url': {'url': img}})

    content.append({'type': 'text', 'text': f'=== TEXT AGENT RESULT ===\n{text_result}'})

    if image_result:
        content.append({'type': 'text', 'text': f'=== IMAGE AGENT RESULT ===\n{image_result}'})
    else:
        content.append({'type': 'text', 'text': '=== IMAGE AGENT RESULT ===\nDiagram section was not submitted.'})

    content.append({'type': 'text', 'text': prompt})

    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{'role': 'user', 'content': content}],
    )

    report = completion.choices[0].message.content

    if human_review_needed:
        report += (
            '\n\n HUMAN REVIEW RECOMMENDED\n'
            'The diagram was flagged as unclear by the Image Agent. '
            'A teacher should manually verify the diagram score before releasing this report.'
        )

    return report

In [6]:

import gradio as gr

# ── State: store agent results between tab runs ───────────────
# Gradio gr.State holds values across button clicks in the same session
text_result_state    = gr.State(None)
text_awarded_state   = gr.State(0)
text_available_state = gr.State(0)
image_result_state   = gr.State(None)
image_awarded_state  = gr.State(0)
image_available_state = gr.State(0)
human_review_state   = gr.State(False)


# ── Agent runner functions ────────────────────────────────────

def grade_text(text_guide_file, student_script_file):
    """Called when teacher clicks Grade Written Test."""
    if text_guide_file is None or student_script_file is None:
        return (
            "Please upload both files.",
            "Upload both files first.",
            None, 0, 0
        )

    result, awarded, available = run_text_agent(
           # Use vision model so it can read handwritten scripts
        guide_file=text_guide_file,
        script_file=student_script_file,
    )

    status = f"Text Agent done — Score: {awarded}/{available}"
    return result, status, result, awarded, available


def grade_image(image_guide_file, student_drawing_file):
    """Called when teacher clicks Grade Diagram."""
    if image_guide_file is None or student_drawing_file is None:
        return (
            "Please upload both files.",
            "Upload both files first.",
            None, 0, 0, False
        )

    result, awarded, available, human_review = run_image_agent(
        guide_file=image_guide_file,
        drawing_file=student_drawing_file,
    )

    status = f"Image Agent done — Score: {awarded}/{available}"
    if human_review:
        status += " ⚠️ Human review recommended"

    return result, status, result, awarded, available, human_review


def generate_final_report(
    text_guide_file,
    t_result, t_awarded, t_available,
    i_result, i_awarded, i_available,
    human_review,
):
    """Called when teacher clicks Generate Final Report."""
    if t_result is None:
        return "Please run Text Grading first before generating the final report."

    report = run_final_boss_agent(
        text_guide_file=text_guide_file,
        text_result=t_result,
        text_awarded=t_awarded,
        text_available=t_available,
        image_result=i_result,
        image_awarded=i_awarded,
        image_available=i_available,
        human_review_needed=human_review,
    )
    return report


# ── Gradio UI ─────────────────────────────────────────────────

with gr.Blocks(title="AI Exam Marking System v3") as app:

    # ── State holders (invisible, store data between clicks) ──
    t_result_s    = gr.State(None)
    t_awarded_s   = gr.State(0)
    t_available_s = gr.State(0)
    i_result_s    = gr.State(None)
    i_awarded_s   = gr.State(0)
    i_available_s = gr.State(0)
    human_s       = gr.State(False)

    # ── Header ──
    gr.Markdown("# AI Exam Marking System")
    gr.Markdown(
        "**How to use:** Upload files in each tab and click Grade. "
        "When both sections are graded, click **Generate Final Report** below."
    )

    # ── Two-tab layout ──
    with gr.Tabs():

        # ════════════════════════════════
        # TAB 1 — TEXT / WRITTEN GRADING
        # ════════════════════════════════
        with gr.Tab("📝 Text / Written Grading"):
            gr.Markdown("### Upload the written test marking guide and the student's written script.")

            with gr.Row():
                text_guide_input = gr.File(
                    label="Marking Guide — Written Test (PDF or Image)",
                    file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                    type="filepath"
                )
                student_script_input = gr.File(
                    label="Student's Written Script (PDF or Image)",
                    file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                    type="filepath"
                )

            text_grade_btn = gr.Button("Grade Written Test", variant="primary", size="lg")
            text_status    = gr.Textbox(label="Status", lines=1, interactive=False)
            text_result_output = gr.Textbox(
                label="Text Grading Result",
                lines=15,
                placeholder="Text grading results will appear here..."
            )

            text_grade_btn.click(
                fn=grade_text,
                inputs=[text_guide_input, student_script_input],
                outputs=[text_result_output, text_status, t_result_s, t_awarded_s, t_available_s]
            )

        # ════════════════════════════════
        # TAB 2 — IMAGE / DIAGRAM GRADING
        # ════════════════════════════════
        with gr.Tab("🖼️ Diagram / Drawing Grading"):
            gr.Markdown("### Upload the reference diagram from the marking guide and the student's drawing.")

            with gr.Row():
                image_guide_input = gr.File(
                    label="Marking Guide — Reference Diagram (PDF or Image)",
                    file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                    type="filepath"
                )
                student_drawing_input = gr.File(
                    label="Student's Drawn Diagram (PDF or Image)",
                    file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                    type="filepath"
                )

            image_grade_btn = gr.Button("Grade Diagram", variant="primary", size="lg")
            image_status    = gr.Textbox(label="Status", lines=1, interactive=False)
            image_result_output = gr.Textbox(
                label="Diagram Grading Result",
                lines=15,
                placeholder="Diagram grading results will appear here..."
            )

            image_grade_btn.click(
                fn=grade_image,
                inputs=[image_guide_input, student_drawing_input],
                outputs=[image_result_output, image_status, i_result_s, i_awarded_s, i_available_s, human_s]
            )

    # ════════════════════════════════
    # FINAL REPORT SECTION
    # ════════════════════════════════
    gr.Markdown("---")
    gr.Markdown("## Final Report")
    gr.Markdown(
        "Once you have graded both sections (or just the text section if there is no diagram), "
        "click below to generate the combined final student report."
    )

    final_report_btn = gr.Button("Generate Final Report", variant="secondary", size="lg")
    final_report_output = gr.Textbox(
        label="Final Student Report",
        lines=25,
        placeholder="The complete student report will appear here..."
    )

    final_report_btn.click(
        fn=generate_final_report,
        inputs=[
            text_guide_input,
            t_result_s, t_awarded_s, t_available_s,
            i_result_s, i_awarded_s, i_available_s,
            human_s
        ],
        outputs=final_report_output
    )

app.launch()

c:\Users\HP\Videos\Github\nitHub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
